# Neurosurgical Site KOL Ranking – Full Pipeline

This notebook runs the complete pipeline end-to-end on a small example dataset.
It is designed to run in **Google Colab** but works equally well in a local Jupyter environment.

## Steps
1. Install dependencies
2. Clone the repository (or upload files)
3. Retrieve PubMed papers
4. Enrich with OpenAlex metadata
5. Classify papers by surgical relevance
6. Build author network & compute metrics
7. Match author affiliations to canonical sites
8. Compute site-level KOL scores
9. Display and download results

---
**API keys** – PubMed E-utilities work without a key at low volume.  
OpenAlex is fully open – no key required.  
The optional LLM step requires an `OPENAI_API_KEY` (set via Colab secrets or environment).


In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
# In Colab the repo is cloned in the next cell; locally just run:
#   pip install -r requirements.txt

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['requests', 'pandas', 'networkx', 'rapidfuzz', 'biopython', 'tqdm']:
    install(pkg)

print('Dependencies installed.')

In [ ]:
# ── 2. Clone the repository (Colab only) ─────────────────────────────────────
import os

REPO_URL = 'https://github.com/roschkoenig/neurosurgical-site-ranking.git'
REPO_DIR = '/content/neurosurgical-site-ranking'

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Optional: set your NCBI API key for higher throughput
# os.environ['NCBI_API_KEY'] = 'YOUR_KEY_HERE'

# Optional: set your email for OpenAlex polite pool
# os.environ['OPENALEX_EMAIL'] = 'you@example.com'

# Optional: set OpenAI key for LLM adjudication
# os.environ['OPENAI_API_KEY'] = 'YOUR_KEY_HERE'

MAX_PUBMED_RESULTS = 50   # small demo; increase to 500+ for real runs
USE_LLM = False           # set True to enable LLM affiliation adjudication

ALIASES_CSV   = 'data/site_aliases.csv'
LONGLIST_CSV  = 'data/site_longlist.csv'

print('Configuration done.')

In [ ]:
# ── 3. Retrieve PubMed papers ────────────────────────────────────────────────
from src.pubmed_search import PubMedSearcher

searcher = PubMedSearcher()
pmids = searcher.search(max_results=MAX_PUBMED_RESULTS)
print(f'Found {len(pmids)} unique PMIDs')

raw_papers = searcher.fetch_details(pmids)
print(f'Fetched details for {len(raw_papers)} papers')

searcher.save_results(raw_papers)
print('Saved raw PubMed results to outputs/pubmed_raw.json')

In [ ]:
# ── 4. Enrich with OpenAlex metadata ─────────────────────────────────────────
from src.openalex_client import OpenAlexClient

client = OpenAlexClient()
papers = client.enrich_papers(raw_papers)
print(f'Enriched {len(papers)} papers with OpenAlex metadata')

In [ ]:
# ── 5. Classify papers by surgical relevance ─────────────────────────────────
import pandas as pd
from src.paper_classifier import PaperClassifier

clf = PaperClassifier()
papers = clf.classify_all(papers)

papers_df = pd.DataFrame([
    {
        'pmid': p.get('pmid', ''),
        'title': p.get('title', ''),
        'journal': p.get('journal', ''),
        'year': p.get('year', ''),
        'doi': p.get('doi', ''),
        'cited_by_count': p.get('cited_by_count', 0),
        'label': p.get('label', ''),
        'paper_weight': p.get('paper_weight', 0),
        'openalex_id': p.get('openalex_id', ''),
    }
    for p in papers
])

print(papers_df['label'].value_counts())
papers_df.to_csv('outputs/papers.csv', index=False)
print('Saved outputs/papers.csv')
papers_df.head()

In [ ]:
# ── 6. Build author network & compute metrics ─────────────────────────────────
from src.author_network import AuthorNetwork

net = AuthorNetwork()
net.build(papers)
authors_df = net.compute_metrics()
print(f'Author network: {len(authors_df)} authors')
authors_df.head(10)

In [ ]:
# ── 7. Match author affiliations to canonical sites ───────────────────────────
from src.site_matcher import SiteMatcher

matcher = SiteMatcher(ALIASES_CSV, use_llm=USE_LLM)
authors_df = matcher.match_authors(authors_df)

print('Match method distribution:')
print(authors_df['match_method'].value_counts())

net.save_authors_csv(authors_df)         # outputs/authors.csv
matcher.save_audit()                     # outputs/match_audit.csv
matcher.save_unresolved(authors_df)      # outputs/unresolved_affiliations.csv

authors_df[authors_df['canonical_site'] != ''].head(10)

In [ ]:
# ── 8. Compute site-level KOL scores ──────────────────────────────────────────
from src.site_scorer import SiteScorer

scorer = SiteScorer()
site_scores = scorer.compute(authors_df)
scorer.save(site_scores)   # outputs/site_scores.csv

print(f'Site scores computed for {len(site_scores)} sites')
site_scores

In [ ]:
# ── 9. Download results (Colab only) ──────────────────────────────────────────
try:
    from google.colab import files
    for fname in ['outputs/papers.csv', 'outputs/authors.csv',
                  'outputs/site_scores.csv', 'outputs/match_audit.csv',
                  'outputs/unresolved_affiliations.csv']:
        try:
            files.download(fname)
        except Exception as e:
            print(f'  Could not download {fname}: {e}')
except ImportError:
    print('Not in Colab – find output files in the outputs/ directory.')